# 44. 제외규정 의미에 따른 HS4 근거 매핑 수정

39번의 계층형 RRF(제품·판정사례 → Baseline 순위 → 추가 근거)를 유지하고, 제외규정의 **출발 HS4(감점)** 와 **대안 HS4(가점)** 를 각각 별도 채널로 구성합니다.

- 제외 원본: IE 레코드의 `관련_HS코드` / `hs_code`
- 대안 코드: IE 레코드의 `관련_분류코드`
- 제33류 주 C33-001: HS4 출발 코드를 특정할 수 없으므로 33류 전체 감점은 하지 않고 명시적 대안 3401만 연결
- 모호한 `제외·예외`, `소호별 포함 범위 및 제외`, C33-009는 자동으로 매핑하지 않음
- 평가 195건, HS4 5종, Outer LOPO 29회, Inner StratifiedGroupKFold 5 folds × 5 seeds, Macro-F1 기준
- 1차 탐색: Product/Ruling/Direct/Alternative/Excluded 채널 각각 [0,0.5,1] (Product+Ruling>0)
- 2차 탐색: 1차 최적점 주변 ±0.25, 0.25 간격
- **Excluded는 음수로만**, Alternative는 양수로만 사용
- 원래 39번 고정 가중치 모델을 별도 비교군으로 동일 계산

이는 규정의 구조를 이용한 규칙 기반 매핑 실험이지 실제 관세사의 법적 품목분류 판정이 아닙니다.


## 실험 목표

기존에는 제외규정에서 `XXXX호` 문자열을 모두 추출하여 하나의 Exclusion 가중치로 결합했습니다. 이는 `근거: 3304호 해설`을 3304의 긍정 근거로 잘못 연결할 수 있습니다.

수정 후에는 명시적으로 `관련_HS코드: 3304` → 감점 채널, `관련_분류코드: 3307` → 가점 채널에 연결합니다. 제외규정의 출발 HS4가 평가 범위에 없으면 감점은 적용하지 않습니다. 가점·감점 모두 해당 코드의 관련 근거 문서가 실제로 존재할 때에만 부여합니다.


## 0. 환경 설정

In [ ]:
!pip -q install -U sentence-transformers scipy kiwipiepy scikit-learn
import re, zipfile, requests
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
WORK=Path('/content/44_hs4_exclusion_semantic_mapping')
WORK.mkdir(parents=True,exist_ok=True)
BASE='https://raw.githubusercontent.com/lin1118-code/HS-complete-product/feature/explanatory-note-integration/'
print(WORK)

## 1. 실험 29와 동일한 데이터 파이프라인

In [ ]:
src=requests.get(BASE+'29_discriminative_features_experiment.ipynb').json()
for i in [3,5,7]: exec(''.join(src['cells'][i]['source']),globals())
print('평가 표본:',len(eval_df),'| HS6:',len(labels),'| 제품군:',eval_df.product_family.nunique(),'| HS4:',eval_df.hs4.nunique())
assert len(eval_df)==195 and len(labels)==11 and eval_df.product_family.nunique()==29 and eval_df.hs4.nunique()==5
hs4_labels=sorted(eval_df.hs4.astype(str).unique())
print('HS4 labels:',hs4_labels)
display(eval_df.groupby('hs4').agg(n=('hs4','size'),families=('product_family','nunique')))
display(pd.crosstab(eval_df.product_family,eval_df.hs4))

## 2. 해설서 근거 채널 생성

In [ ]:
kb=pd.read_json(BASE+'hs_knowledge_base.jsonl',lines=True).fillna('')
for c in ['knowledge_id','hs_code','rule_type','document_text','source_sheet','source_page_or_line']:
    if c not in kb: kb[c]=''
    kb[c]=kb[c].astype(str).str.strip()
kb['hs_digits']=kb.hs_code.map(lambda x:re.sub(r'\D','',x))
def clean_kb_text(x):
    lines=[]
    for line in str(x).splitlines():
        if re.match(r'^(규칙_ID|HS코드|자료출처|류|적용범위)\s*:',line.strip()): continue
        lines.append(line.strip())
    return ' '.join(x for x in lines if x)
kb['embedding_text']=kb.document_text.map(clean_kb_text)
direct_rows=[]
exclusion_rows=[]
for _,r in kb.iterrows():
    typ=r.rule_type
    own=r.hs_digits[:4]
    if own in hs4_labels and r.embedding_text and '제외' not in typ:
        direct_rows.append({**r.to_dict(),'candidate_hs4':own,'channel':'direct'})
    if '제외' in typ and r.embedding_text:
        refs=sorted(set(re.findall(r'(?:제\s*)?(\d{4})\s*호',r.document_text)))
        for h in refs:
            if h[:4] in hs4_labels: exclusion_rows.append({**r.to_dict(),'candidate_hs4':h[:4],'channel':'exclusion'})
direct_docs=pd.DataFrame(direct_rows).drop_duplicates(['knowledge_id','candidate_hs4']).reset_index(drop=True)
exclusion_docs=pd.DataFrame(exclusion_rows).drop_duplicates(['knowledge_id','candidate_hs4']).reset_index(drop=True)
print('KB:',len(kb),'| Direct:',len(direct_docs),'| Exclusion links:',len(exclusion_docs))
display(pd.concat([direct_docs.groupby('candidate_hs4').size().rename('direct_docs'),exclusion_docs.groupby('candidate_hs4').size().rename('exclusion_links')],axis=1).reindex(hs4_labels).fillna(0).astype(int))
display(exclusion_docs[['knowledge_id','candidate_hs4','rule_type','source_page_or_line','document_text']])

## 3. 임베딩 생성 및 캐시

In [ ]:
CACHE=WORK/'embeddings_44_base.npz'
if CACHE.exists():
    z=np.load(CACHE,allow_pickle=True)
    query_emb,product_emb,ruling_emb,kb_direct_emb,kb_exclusion_emb=z['query_emb'],z['product_emb'],z['ruling_emb'],z['kb_direct_emb'],z['kb_exclusion_emb']
else:
    sbert=SentenceTransformer('intfloat/multilingual-e5-base')
    def encode_query(x): return sbert.encode(['query: '+str(v) for v in x],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
    def encode_passage(x): return sbert.encode(['passage: '+str(v) for v in x],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
    query_emb=encode_query(eval_df.query_text.tolist())
    product_emb=encode_passage(eval_df.query_text.tolist())
    ruling_emb=encode_passage(eval_df.ruling_text.tolist())
    kb_direct_emb=encode_passage(direct_docs.embedding_text.tolist()) if len(direct_docs) else np.empty((0,768),dtype=float)
    kb_exclusion_emb=encode_passage(exclusion_docs.embedding_text.tolist()) if len(exclusion_docs) else np.empty((0,768),dtype=float)
    np.savez_compressed(CACHE,query_emb=query_emb,product_emb=product_emb,ruling_emb=ruling_emb,kb_direct_emb=kb_direct_emb,kb_exclusion_emb=kb_exclusion_emb)
print(query_emb.shape,product_emb.shape,ruling_emb.shape,kb_direct_emb.shape,kb_exclusion_emb.shape)

## 4. 순위 함수와 고정 검색 캐시

In [ ]:
def aggregate_hs4(scores,codes,method='top3_mean'):
    d=pd.DataFrame({'hs4':np.asarray(codes).astype(str),'score':np.asarray(scores,dtype=float)})
    if method=='max': out=d.groupby('hs4').score.max()
    elif method=='mean': out=d.groupby('hs4').score.mean()
    else: out=d.groupby('hs4').score.apply(lambda x:x.nlargest(min(3,len(x))).mean())
    return out.sort_values(ascending=False)
def rrf_two(a,b,k=60,w=1.0):
    pa={str(c):i+1 for i,c in enumerate(a.index)}
    pb={str(c):i+1 for i,c in enumerate(b.index)}
    return pd.Series({c:1/(k+pa.get(c,99999))+w/(k+pb.get(c,99999)) for c in hs4_labels}).sort_values(ascending=False)
def rrf_ablation(base,direct=None,exclusion=None,wd=0.0,we=0.0,k=60):
    ranks=[base]
    weights=[1.0]
    if direct is not None: ranks.append(direct); weights.append(wd)
    if exclusion is not None: ranks.append(exclusion); weights.append(we)
    pos=[{str(c):i+1 for i,c in enumerate(r.index)} for r in ranks]
    return pd.Series({c:sum(w/(k+p.get(c,99999)) for w,p in zip(weights,pos)) for c in hs4_labels}).sort_values(ascending=False)
def rank_kb(scores,docs,method):
    vals={}
    for c in hs4_labels:
        x=np.asarray(scores)[docs.candidate_hs4.astype(str).to_numpy()==c]
        if len(x)==0: vals[c]=-1.0
        elif method=='max': vals[c]=float(np.max(x))
        else: vals[c]=float(np.mean(np.sort(x)[-min(3,len(x)):]))
    return pd.Series(vals).sort_values(ascending=False)
def baseline_rank(train_idx,q):
    codes=eval_df.iloc[train_idx].hs4.astype(str).to_numpy()
    rp=aggregate_hs4(product_emb[train_idx]@query_emb[q],codes,'top3_mean')
    rr=aggregate_hs4(ruling_emb[train_idx]@query_emb[q],codes,'top3_mean')
    return rrf_two(rp,rr,w=1.0)
METHODS=['max','top3_mean']
WEIGHTS=[0.0,0.5,1.0,2.0]
direct_rank_cache={}
exclusion_rank_cache={}
for q in range(len(eval_df)):
    ds=kb_direct_emb@query_emb[q] if len(direct_docs) else np.array([])
    es=kb_exclusion_emb@query_emb[q] if len(exclusion_docs) else np.array([])
    direct_rank_cache[q]={m:rank_kb(ds,direct_docs,m) for m in METHODS}
    exclusion_rank_cache[q]={m:rank_kb(es,exclusion_docs,m) for m in METHODS}
splits=list(LeaveOneGroupOut().split(eval_df,eval_df.hs4,groups=eval_df.product_family))
print('Outer LOPO:',len(splits))
print('Direct docs:',len(direct_docs),'| Exclusion links:',len(exclusion_docs))

In [ ]:
def read_field(text,name):
    m=re.search(r'^'+re.escape(name)+r'\s*:\s*(.*)$',str(text),flags=re.MULTILINE)
    return m.group(1).strip() if m else ''
def explicit_hs4(text):
    return sorted(set(re.findall(r'(?<!\d)(\d{4})(?!\d)',text)))
positive_rows=[]
negative_rows=[]
mapping_audit=[]
for _,r in kb.iterrows():
    rid=str(r.knowledge_id)
    typ=str(r.rule_type)
    if not typ=='제외': continue
    source=read_field(r.document_text,'관련_HS코드')
    target=read_field(r.document_text,'관련_분류코드')
    if rid.startswith('IE-') and source and target:
        s_codes=[c for c in explicit_hs4(source) if c in hs4_labels]
        t_codes=[c for c in explicit_hs4(target) if c in hs4_labels]
        status='mapped' if (s_codes or t_codes) else 'outside_scope'
        for c in s_codes:
            negative_rows.append({**r.to_dict(),'candidate_hs4':c,'channel':'excluded_from'})
        for c in t_codes:
            positive_rows.append({**r.to_dict(),'candidate_hs4':c,'channel':'alternative_to'})
        mapping_audit.append({'knowledge_id':rid,'source_codes':'|'.join(s_codes),'alternative_codes':'|'.join(t_codes),'status':status,'source_raw':source,'alternative_raw':target})
    elif rid=='C33-001':
        alternatives=[c for c in re.findall(r'제\s*(\d{4})\s*호',r.document_text) if c in hs4_labels]
        for c in sorted(set(alternatives)):
            positive_rows.append({**r.to_dict(),'candidate_hs4':c,'channel':'alternative_to'})
        mapping_audit.append({'knowledge_id':rid,'source_codes':'','alternative_codes':'|'.join(sorted(set(alternatives))),'status':'chapter_note_alternative_only','source_raw':'제33류','alternative_raw':'|'.join(alternatives)})
    else:
        mapping_audit.append({'knowledge_id':rid,'source_codes':'','alternative_codes':'','status':'skipped_ambiguous','source_raw':'','alternative_raw':''})
positive_docs=pd.DataFrame(positive_rows).drop_duplicates(['knowledge_id','candidate_hs4']).reset_index(drop=True)
negative_docs=pd.DataFrame(negative_rows).drop_duplicates(['knowledge_id','candidate_hs4']).reset_index(drop=True)
mapping_audit=pd.DataFrame(mapping_audit)
display(mapping_audit)
display(pd.concat([positive_docs.groupby('candidate_hs4').size().rename('alternative'),negative_docs.groupby('candidate_hs4').size().rename('excluded')],axis=1).reindex(hs4_labels).fillna(0).astype(int))
assert '3307' in positive_docs.loc[positive_docs.knowledge_id=='IE-014','candidate_hs4'].tolist()
assert '3304' in negative_docs.loc[negative_docs.knowledge_id=='IE-014','candidate_hs4'].tolist()
assert '3304' not in positive_docs.loc[positive_docs.knowledge_id=='IE-014','candidate_hs4'].tolist()


In [ ]:
SEMANTIC_CACHE=WORK/'44_semantic_embeddings.npz'
if SEMANTIC_CACHE.exists():
    data=np.load(SEMANTIC_CACHE)
    alt_emb=data['alt_emb']; excl_emb=data['excl_emb']
else:
    model=SentenceTransformer('intfloat/multilingual-e5-base')
    def kb_encode(df):
        if not len(df):return np.empty((0,query_emb.shape[1]),dtype=np.float32)
        texts=['passage: '+str(t) for t in df.embedding_text]
        return model.encode(texts,batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
    alt_emb=kb_encode(positive_docs)
    excl_emb=kb_encode(negative_docs)
    np.savez_compressed(SEMANTIC_CACHE,alt_emb=alt_emb,excl_emb=excl_emb)
print('Semantic docs:',alt_emb.shape,excl_emb.shape)


## 5. 반복 Stratified Group K-Fold Inner 튜닝

In [ ]:
INNER_SEEDS=[11,23,37,53,71]
INNER_SPLITS=5
LABELS=np.array(hs4_labels,dtype=str)
COARSE=np.array([0.0,0.5,1.0])
GRID=np.array([(p,r,d,a,x) for p in COARSE for r in COARSE for d in COARSE for a in COARSE for x in COARSE if p+r>0],dtype=np.float64)
FIXED_39=np.array([1.,1.,0.5,0.5])
BASELINE=np.array([1.,1.,0.,0.,0.])
alt_rank_cache={};excl_rank_cache={}
def semantic_rank(q,docs,emb):
    if not len(docs):return np.zeros(len(LABELS),dtype=float)
    scores=emb@query_emb[q]
    rank=rank_kb(scores,docs,'max')
    available=set(docs.candidate_hs4.astype(str))
    ordered=list(rank.index.astype(str))
    return np.array([1/(60+ordered.index(c)+1) if c in available else 0.0 for c in LABELS])
for q in range(len(eval_df)):
    alt_rank_cache[q]=semantic_rank(q,positive_docs,alt_emb)
    excl_rank_cache[q]=semantic_rank(q,negative_docs,excl_emb)
def rank_vector(series):
    order=list(series.index.astype(str))
    return np.array([1/(60+order.index(c)+1) if c in order else 1/(60+99999) for c in LABELS])
def build_features(train_idx,q):
    codes=eval_df.iloc[train_idx].hs4.astype(str).to_numpy()
    ps=aggregate_hs4(product_emb[train_idx]@query_emb[q],codes,'top3_mean')
    rs=aggregate_hs4(ruling_emb[train_idx]@query_emb[q],codes,'top3_mean')
    dr=rank_vector(direct_rank_cache[q]['top3_mean'])
    return np.stack([rank_vector(ps),rank_vector(rs),dr,alt_rank_cache[q],excl_rank_cache[q]])
def score_weights(X,W):
    first=np.einsum('wc,nck->nwk',W[:,:2],X[:,:2,:],optimize=True)
    order=np.argsort(-first,axis=2,kind='stable')
    ranks=np.empty_like(order)
    np.put_along_axis(ranks,order,np.arange(1,len(LABELS)+1)[None,None,:],axis=2)
    baseline=1/(60+ranks)
    return baseline+np.einsum('wc,nck->nwk',W[:,2:4],X[:,2:4,:],optimize=True)-W[:,4][None,:,None]*X[:,4,None,:]
def predict_weights(X,W):
    return np.argmax(score_weights(X,W),axis=2)
def evaluate_weights(X,y,W):
    predictions=predict_weights(X,W)
    accuracy=(predictions==y[:,None]).mean(axis=0)
    f1=np.zeros(len(W))
    for c in range(len(LABELS)):
        tp=((predictions==c)&(y[:,None]==c)).sum(axis=0)
        fp=((predictions==c)&(y[:,None]!=c)).sum(axis=0)
        fn=((predictions!=c)&(y[:,None]==c)).sum(axis=0)
        f1+=np.divide(2*tp,2*tp+fp+fn,out=np.zeros(len(W)),where=(2*tp+fp+fn)>0)/len(LABELS)
    return f1,accuracy
def inner_data(outer_train):
    inner=eval_df.iloc[outer_train].reset_index(drop=True)
    y=inner.hs4.astype(str).to_numpy()
    groups=inner.product_family.to_numpy()
    repeats=[];coverage=[]
    for seed in INNER_SEEDS:
        splitter=StratifiedGroupKFold(n_splits=INNER_SPLITS,shuffle=True,random_state=seed)
        feat=[];truth=[];validated=set();used=0
        for itr,iva in splitter.split(inner,y,groups):
            gtr,gva=outer_train[itr],outer_train[iva]
            if set(eval_df.iloc[gtr].hs4.astype(str))!=set(hs4_labels):continue
            used+=1
            for q in gva:
                feat.append(build_features(gtr,q))
                truth.append(hs4_labels.index(str(eval_df.iloc[q].hs4)))
                validated.add(int(q))
        if not feat:raise ValueError(f'Inner seed {seed}: no valid split')
        repeats.append((np.stack(feat),np.array(truth,dtype=int)))
        coverage.append({'seed':seed,'valid_splits':used,'validation_rows':len(validated),'coverage_rate':len(validated)/len(outer_train)})
    return repeats,coverage
def score_grid(repeats,weights):
    fs=[];acc=[]
    for X,y in repeats:
        f,a=evaluate_weights(X,y,weights)
        fs.append(f);acc.append(a)
    return np.mean(fs,axis=0),np.mean(acc,axis=0)
def pick(repeats,weights):
    f,a=score_grid(repeats,weights)
    idx=int(np.lexsort((-a,-f))[0])
    return weights[idx],float(f[idx]),float(a[idx])
def fine_grid(center):
    axes=[]
    for i in range(5):
        axes.append(np.round(np.arange(max(0,center[i]-0.25),min(1,center[i]+0.25)+0.001,0.25),2))
    return np.array([(p,r,d,a,x) for p in axes[0] for r in axes[1] for d in axes[2] for a in axes[3] for x in axes[4] if p+r>0],dtype=np.float64)
print('Coarse candidates:',len(GRID),'Outer folds:',len(splits))


## 6. Outer LOPO 실행 (5개 Fold씩)

In [ ]:
FOLD_START=1
FOLD_END=5
def fold_path(f):return WORK/f'fold_{f:02d}.csv'
def tuning_path(f):return WORK/f'fold_{f:02d}_tuning.csv'
def coverage_path(f):return WORK/f'fold_{f:02d}_coverage.csv'
for fold,(tr,te) in enumerate(splits,1):
    if not FOLD_START<=fold<=FOLD_END:continue
    if set(eval_df.iloc[tr].hs4.astype(str))!=set(hs4_labels):raise ValueError('Outer train missing class')
    if all(p.exists() for p in [fold_path(fold),tuning_path(fold),coverage_path(fold)]):
        print('Fold',fold,'완료 - skip');continue
    repeats,coverage=inner_data(tr)
    coarse,cf,ca=pick(repeats,GRID)
    fine,ff,fa=pick(repeats,fine_grid(coarse))
    best=coarse if (cf,ca)>=(ff,fa) else fine
    rows=[]
    for q in te:
        X=build_features(tr,q)[None,:,:]
        original_base=baseline_rank(tr,q)
        original_fixed=rrf_ablation(original_base,direct_rank_cache[q]['top3_mean'],exclusion_rank_cache[q]['max'],0.5,0.5)
        for model,w in [('Baseline_39',BASELINE),('Original_Fixed39',None),('Semantic_Coarse',coarse),('Semantic_Fine',fine),('Semantic_Selected',best)]:
            if model=='Baseline_39':prediction=str(original_base.index[0])
            elif model=='Original_Fixed39':prediction=str(original_fixed.index[0])
            else:prediction=str(LABELS[int(predict_weights(X,w[None,:])[0,0])])
            row={'row_index':int(q),'fold':fold,'heldout_family':str(eval_df.iloc[q].product_family),'true_hs4':str(eval_df.iloc[q].hs4),'model':model,'predicted_hs4':prediction}
            if w is not None:row.update(dict(zip(['product_weight','ruling_weight','direct_weight','alternative_weight','excluded_penalty_weight'],map(float,w))))
            rows.append(row)
    pd.DataFrame(rows).to_csv(fold_path(fold),index=False,encoding='utf-8-sig')
    pd.DataFrame([{'fold':fold,'stage':stage,'product':float(w[0]),'ruling':float(w[1]),'direct':float(w[2]),'alternative':float(w[3]),'excluded_penalty':float(w[4]),'inner_macro_f1':f,'inner_accuracy':a} for stage,w,f,a in [('coarse',coarse,cf,ca),('fine',fine,ff,fa),('selected',best,*( (cf,ca) if np.array_equal(best,coarse) else (ff,fa) ))]]).to_csv(tuning_path(fold),index=False,encoding='utf-8-sig')
    pd.DataFrame(coverage).assign(outer_fold=fold).to_csv(coverage_path(fold),index=False,encoding='utf-8-sig')
    print(f'Fold {fold}/{len(splits)} | selected={best.tolist()} | inner_F1={max(cf,ff):.4f}')
print('구간 완료')


## 7. 진행률 및 중간 백업

In [ ]:
done=sorted(int(p.stem.split('_')[1]) for p in WORK.glob('fold_[0-9][0-9].csv'))
print('완료:',len(done),'/',len(splits),'| 누락:',sorted(set(range(1,len(splits)+1))-set(done)))
BACKUP='/content/44_fold_backup.zip'
with zipfile.ZipFile(BACKUP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in WORK.glob('fold_*.csv'):z.write(p,p.name)
    if CACHE.exists():z.write(CACHE,CACHE.name)
    if SEMANTIC_CACHE.exists():z.write(SEMANTIC_CACHE,SEMANTIC_CACHE.name)
from google.colab import files
files.download(BACKUP)


## 8. 중간 백업 복구 (새 런타임에서만 실행)

In [ ]:
from google.colab import files
uploaded=files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z: z.extractall(WORK)
        print('복구 완료:',name)

## 9. 통합 평가 및 최종 결과 ZIP

In [ ]:
files_list=sorted(WORK.glob('fold_[0-9][0-9].csv'))
missing=sorted(set(range(1,len(splits)+1))-set(int(p.stem.split('_')[1]) for p in files_list))
print('누락:',missing)
if missing:raise ValueError('미완료 Fold가 있습니다.')
pred=pd.concat([pd.read_csv(p,dtype={'true_hs4':str,'predicted_hs4':str}) for p in files_list],ignore_index=True)
if pred.duplicated(['row_index','model']).any():raise ValueError('중복 예측 존재')
summary=pred.groupby('model').apply(lambda g:pd.Series({'n':len(g),'macro_f1':f1_score(g.true_hs4,g.predicted_hs4,labels=hs4_labels,average='macro',zero_division=0),'accuracy':accuracy_score(g.true_hs4,g.predicted_hs4)}),include_groups=False).reset_index().sort_values('macro_f1',ascending=False)
tuning=pd.concat([pd.read_csv(p) for p in sorted(WORK.glob('fold_[0-9][0-9]_tuning.csv'))],ignore_index=True)
coverage=pd.concat([pd.read_csv(p) for p in sorted(WORK.glob('fold_[0-9][0-9]_coverage.csv'))],ignore_index=True)
OUT=WORK/'final';OUT.mkdir(exist_ok=True)
for name,df in [('44_predictions',pred),('44_model_comparison',summary),('44_selected_weights',tuning),('44_coverage',coverage)]:
    df.to_csv(OUT/f'{name}.csv',index=False,encoding='utf-8-sig')
mapping_audit.to_csv(OUT/'44_rule_mapping_audit.csv',index=False,encoding='utf-8-sig')
positive_docs.to_csv(OUT/'44_alternative_documents.csv',index=False,encoding='utf-8-sig')
negative_docs.to_csv(OUT/'44_excluded_documents.csv',index=False,encoding='utf-8-sig')
display(summary.round(4))
display(tuning[tuning.stage=='selected'].round(3))
with zipfile.ZipFile('/content/44_hs4_hierarchical_weight_search_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob('*.csv'):z.write(p,p.name)
from google.colab import files
files.download('/content/44_hs4_hierarchical_weight_search_results.zip')


## 결과 해석

1. `Original_Fixed39`는 39번 원래 Exclusion 문서 매핑을 재사용한 고정 모델입니다.
2. `Semantic_Selected`는 제외 대상(감점)과 대안 코드(가점)를 분리하여 Inner CV에서 튜닝한 모델입니다.
3. 클래스별 성능과 `44_rule_mapping_audit.csv`에서 어떤 근거가 실제로 연결됐는지를 검토하세요.
4. 가중치 검색의 동점 처리 차이와 불균형 제품군 분할로 인한 검증 불안정성은 남아 있습니다. 새 모델의 개선은 결과 확인 전까지 주장하지 않습니다.
